In [2]:
import duckdb
import pandas as pd

# -----------------------------
# CONNECT TO DB
# -----------------------------
conn = duckdb.connect(
    "../database/airport_ops.duckdb"
)

# -----------------------------
# AIRPORT METRICS
# -----------------------------
airport_metrics = conn.sql("""
SELECT

    a.iata_code,
    a.name AS airport_name,
    a.iso_country,
    a.municipality,
    a.latitude_deg,
    a.longitude_deg,

    w.temperature_c,
    w.wind_kph,
    w.visibility_km,
    w.weather_condition,

    COUNT(f.icao24)
        AS nearby_flights

FROM airports a

LEFT JOIN weather w
ON TRIM(a.iata_code)
=
TRIM(w.airport_code)

LEFT JOIN flights f
ON ABS(
    a.latitude_deg
    - f.latitude
) < 1

AND ABS(
    a.longitude_deg
    - f.longitude
) < 1

WHERE a.iata_code
IS NOT NULL

GROUP BY ALL
""").df()

# -----------------------------
# CREATE STRESS SCORE
# -----------------------------
airport_metrics[
    "weather_risk"
] = 0

airport_metrics.loc[
    airport_metrics[
        "wind_kph"
    ] > 40,
    "weather_risk"
] += 30

airport_metrics.loc[
    airport_metrics[
        "visibility_km"
    ] < 5,
    "weather_risk"
] += 30

airport_metrics[
    "airport_stress_score"
] = (
    airport_metrics[
        "nearby_flights"
    ] * 0.6
    +
    airport_metrics[
        "weather_risk"
    ] * 0.4
)

# -----------------------------
# SAVE TABLE
# -----------------------------
conn.register(
    "metrics_df",
    airport_metrics
)

conn.execute("""
CREATE OR REPLACE TABLE
airport_metrics AS
SELECT *
FROM metrics_df
""")

print(
    "Airport metrics created!"
)

airport_metrics.head()

Airport metrics created!


,iata_code,airport_name,iso_country,municipality,latitude_deg,longitude_deg,temperature_c,wind_kph,visibility_km,weather_condition,nearby_flights,weather_risk,airport_stress_score
0,PRN,Priština Adem Jashari International Airport,XK,Prishtina,42.572800,21.035801,14.1,10.4,10.0,Patchy rain nearby,12,0,7.2
1,YAZ,Tofino / Long Beach Airport,CA,Tofino,49.079833,-125.775583,11.7,7.6,10.0,Partly Cloudy,4,0,2.4
2,QBC,Bella Coola Airport,CA,Bella Coola,52.387501,-126.596001,8.2,3.6,10.0,Sunny,2,0,1.2
3,YBG,Saguenay-Bagotville Airport,CA,Saguenay,48.330123,-70.992012,13.6,29.2,10.0,Sunny,1,0,0.6
4,YBL,Campbell River Airport,CA,Campbell River,49.950802,-125.271004,13.8,7.9,10.0,Sunny,8,0,4.8


In [3]:
conn.sql("""
SELECT *
FROM airport_metrics
ORDER BY
airport_stress_score DESC
LIMIT 200
""").df()

,iata_code,airport_name,iso_country,municipality,latitude_deg,longitude_deg,temperature_c,wind_kph,visibility_km,weather_condition,nearby_flights,weather_risk,airport_stress_score
0,PHX,Phoenix Sky Harbor International Airport,US,Phoenix,33.435302,-112.005905,28.2,4.7,10.0,Sunny,249,0,149.4
1,LUF,Luke Air Force Base,US,Glendale,33.535000,-112.383003,28.1,8.6,10.0,Sunny,244,0,146.4
2,MMU,Morristown Municipal Airport,US,Morristown,40.799062,-74.414949,31.8,19.8,10.0,Sunny,236,0,141.6
3,WRI,Mc Guire Air Force Base,US,Wrightstown,40.015598,-74.591698,32.7,22.7,10.0,Sunny,235,0,141.0
4,TEB,Teterboro Airport,US,Teterboro,40.850101,-74.060799,33.0,21.6,10.0,Sunny,235,0,141.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,RIC,Richmond International Airport,US,Richmond,37.505199,-77.319702,33.3,12.6,10.0,Sunny,78,0,46.8
196,LWM,Lawrence Municipal Airport,US,Lawrence,42.717201,-71.123398,29.7,24.8,10.0,Partly Cloudy,78,0,46.8
197,NRB,Naval Station Mayport / Admiral David L McDona...,US,Jacksonville,30.391100,-81.424698,27.7,15.8,10.0,Patchy rain nearby,78,0,46.8
198,BAB,Beale Air Force Base,US,Beale Air Force Base,39.136101,-121.436996,23.9,4.3,10.0,Sunny,78,0,46.8
